# 점수 예측

학습 데이터 설치

In [4]:
import numpy as np
import pandas as pd
import sqlite3 as sql
import os

In [2]:
import kagglehub

path = kagglehub.dataset_download("wyattowalsh/basketball", output_dir="dataset/")

print("Path to dataset files:", path)

c:\Users\jaehy\.conda\envs\dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 697M/697M [01:05<00:00, 11.2MB/s] 

Extracting files...


Path to dataset files: dataset/


In [6]:
db_path = 'dataset/nba.sqlite'
connection = sql.connect(db_path) # create connection object to database
print("SQL database connected")
table = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", connection)
print(table)

SQL database connected
                   name
0                  game
1          game_summary
2           other_stats
3             officials
4      inactive_players
5             game_info
6            line_score
7          play_by_play
8                player
9                  team
10   common_player_info
11         team_details
12         team_history
13  draft_combine_stats
14        draft_history
15     team_info_common


In [9]:
q = "SELECT game_id, eventnum, period, pctimestring, homedescription,neutraldescription,visitordescription FROM play_by_play LIMIT 1000"
df = pd.read_sql_query(q, connection)

In [11]:
print(df.head(50))

       game_id  eventnum  period pctimestring  \
0   0029600012         0       1        12:00   
1   0029600012         2       1        12:00   
2   0029600012         3       1        11:45   
3   0029600012         4       1        11:43   
4   0029600012         5       1        11:29   
5   0029600012         6       1        11:27   
6   0029600012         7       1        11:14   
7   0029600012         8       1        11:08   
8   0029600012         9       1        10:49   
9   0029600012        10       1        10:49   
10  0029600012        11       1        10:45   
11  0029600012        12       1        10:42   
12  0029600012        13       1        10:40   
13  0029600012        14       1        10:19   
14  0029600012        15       1        10:18   
15  0029600012        16       1        10:05   
16  0029600012        17       1         9:42   
17  0029600012        18       1         9:40   
18  0029600012        19       1         9:38   
19  0029600012      

학습 데이터 전처리

In [ ]:
def parse_clock_to_sec(clock:str) -> int:
    minutes, seconds = clock.split(':')
    return int(minutes) * 60 + int(seconds)

In [ ]:
def merge_event_description(row: pd.Series) -> str:
    home_desc = row.get('homedescription', None)
    neutral_desc = row.get('neutraldescription', None)
    visitor_desc = row.get('visitordescription', None)
    if pd.notna(home_desc):
        return home_desc
    elif pd.notna(visitor_desc):
        return visitor_desc
    elif pd.notna(neutral_desc):
        return neutral_desc
    return ""

In [12]:
def extract_event_type(desc: str) -> str:
    if pd.isna(desc) or desc == "":
        return "unknown"
    #todo: more event types

### Model

In [13]:
import torch
import torch.nn as nn

In [14]:
class ScorePrediction(nn.Module):
  def __init__(self,
               num_event_types,
               num_numeric_features,
               hidden_size,
               num_layers,
               event_emb_dim,
               period_emb_dim=4,
               team_emb_dim=2,
               output_size=2
              ):
    super(ScorePrediction, self).__init__()

    self.event_emb = nn.Embedding(num_event_types, event_emb_dim)
    self.period_emb = nn.Embedding(4, period_emb_dim)
    self.team_emb = nn.Embedding(2, team_emb_dim)

    input_dim = event_emb_dim + period_emb_dim + team_emb_dim + num_numeric_features

    self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=num_layers, batch_first=True)

    self.regressor = nn.Sequential(
      nn.Linear(hidden_size, 64),
      nn.ReLU(),
      nn.Linear(64, output_size)
    )

  def forward(self, event_type_ids, period_type_ids, team_ids, numeric_feats):

    event_e = self.event_emb(event_type_ids)
    period_e = self.period_emb(period_type_ids)
    team_e = self.team_emb(team_ids)

    x = torch.cat([event_e,period_e,team_e], dim=-1)

    out, (h_n, c_n) = self.lstm(x)

    out = out[:,-1,:]

    pred = self.regressor(out)

    return pred
